# 🔥 NEVER-AI (`ChronoMorph-LM`) — Super-Entraîneur GPU Cloud + Google Drive

Ce notebook entraîne **votre propre architecture `ChronoMorph-LM` « From Scratch »** sur un vrai **GPU NVIDIA T4 / A100** (50× à 100× plus rapide qu'un CPU) sur des **millions de vrais textes humains francophones & anglophones** (`Wikipedia FR`, `OpenHermes`, dialogues naturels), puis sauvegarde directement vos poids `.safetensors` dans votre **Google Drive** !

> **Avant de lancer :** Dans le menu Colab en haut, vérifiez **Exécution → Modifier le type d'exécution → GPU T4**.

In [ ]:
# 1. Montage de votre Google Drive pour sauvegarder les poids neuronaux en continu
from google.colab import drive
drive.mount('/content/drive')
DRIVE_VAULT = '/content/drive/MyDrive/NeverAI_Vault/checkpoints'
import os
os.makedirs(DRIVE_VAULT, exist_ok=True)
print('Coffre Google Drive prêt :', DRIVE_VAULT)

In [ ]:
# 2. Clonage du dépôt Never-AI et installation des dépendances GPU + HuggingFace Datasets
!git clone -b arena/403f5d7f-never-ai https://github.com/mouusieur/never-ai.git /content/never-ai || (cd /content/never-ai && git pull)
%cd /content/never-ai
!pip install -q -e . datasets

In [ ]:
# 3. Téléchargement en streaming d'un vrai corpus massif FR + Dialogues naturels (zéro template robotique)
from datasets import load_dataset
import torch
print('GPU actif :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

real_texts = []
print('Chargement en streaming de Wikipedia FR + instructions conversationnelles...')
wiki_fr = load_dataset('wikimedia/wikipedia', '20231101.fr', split='train', streaming=True)
for i, row in enumerate(wiki_fr):
    if i >= 12000:
        break
    txt = row['text'].strip()
    if len(txt) > 300:
        real_texts.append(f"<|bos|><|user|>Explique clairement : {row['title']}<|never|>{txt[:1800]}<|eos|>")

print(f'Corpus réel assemblé : {len(real_texts):,} documents humains riches !')

In [ ]:
# 4. Entraînement intensif GPU (CompositeMuon + WSM + Croissance Morphogénétique Sprout-MoE -> Canopy-MoE)
from never_ai.core.config import ChronoMorphConfig
from never_ai.core.model import ChronoMorphLM
from never_ai.core.tokenizer import MorphTokenizer
from never_ai.data.curriculum import ContinualDatasetAndReplay
from never_ai.optim.composite_muon import CompositeMuonOptimizer, WSMCheckpointMerger
from never_ai.morphogenesis.operators import transition_seed0_to_seed1, transition_seed1_to_sprout_moe
from never_ai.vault.manager import VaultManager
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vault = VaultManager(DRIVE_VAULT)

# Vocabulaire BPE élargi à 16 384 tokens pour une fluidité française maximale !
tokenizer = MorphTokenizer.train_from_iterator(real_texts[:6000], vocab_size=16384)
cfg = ChronoMorphConfig.seed_0(vocab_size=tokenizer.vocab_size)
model = ChronoMorphLM(cfg).to(device)
model = transition_seed0_to_seed1(model).to(device)
model = transition_seed1_to_sprout_moe(model, n_routed_experts=8, top_k=2).to(device)
print('Modèle initialisé sur GPU :', model.count_parameters())

dataset = ContinualDatasetAndReplay(tokenizer, seq_len=256)
dataset.load_corpora(real_texts[:-200], real_texts[-200:])
opt = CompositeMuonOptimizer(model, muon_lr=4e-3, adamw_lr=1e-3, warmup_steps=100)
wsm = WSMCheckpointMerger(max_snapshots=6)

STEPS = 3000  # Augmentez à 20_000+ pour un entraînement profond de plusieurs heures !
t0 = time.time()
model.train()
for step in range(1, STEPS + 1):
    x, y = dataset.sample_batch(batch_size=16, device=device)
    opt.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
        out = model(x, targets=y)
        loss = out['loss']
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 100 == 0:
        wsm.record_snapshot(model)
        print(f'[Step {step}/{STEPS}] loss={loss.item():.4f} | elapsed={time.time()-t0:.1f}s')
    if step % 500 == 0:
        vault.save_checkpoint('latest', model, tokenizer, metrics={'val_ce_loss': float(loss.item())}, consolidated_state_dict=wsm.merge())
        print('  -> Sauvegardé automatiquement sur Google Drive !')

vault.save_checkpoint('latest', model, tokenizer, metrics={'val_ce_loss': float(loss.item())}, consolidated_state_dict=wsm.merge())
print('✅ Entraînement GPU terminé ! Copiez le dossier NeverAI_Vault/checkpoints/latest dans votre installation Never-AI.')